In [ ]:
"""
Generate all figures for the Section 7 of the HMKF paper.

Produces:
  1. hd_distribution.pdf      – histogram + CDF of HD across 130 in-scope systems
  2. gt_recon_success.pdf     – 3-panel GT vs recon, good systems
  3. gt_recon_failure.pdf     – 3-panel GT vs recon, failure examples
  4. full_benchmark_mosaic.pdf – mosaic of all 130 in-scope systems (2D projections)
  5. figures/GT_vs_recon_{system}.pdf – per-system GT vs recon (all 130)

Usage:
  Set DRIVE_ROOT and OUT_DIR below, then run all cells!
"""



# ── Imports ───────────────────────────────────────────────────────────────────
import os, csv, math, statistics
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from mpl_toolkits.mplot3d import Axes3D          # noqa: F401
from pathlib import Path


# ── Config ────────────────────────────────────────────────────────────────────
PROJECT_ROOT  = "."
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
DATA_DIR      = os.path.join(RESULTS_DIR, "trajectories")
OUT_DIR       = "figures"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
NPZ_DIR     = "results/trajectories"
RESULTS_CSV = "results/results.csv"
OUT_DIR     = "figures"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(f"{OUT_DIR}/per_system", exist_ok=True)

# ── Style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
GT_COLOR    = "#888888"   # grey – ground truth
RECON_COLOR = "#C0392B"   # dark red – reconstructed set

# ── Load results ──────────────────────────────────────────────────────────────
with open(RESULTS_CSV) as f:
    rows = {r["System"]: r for r in csv.DictReader(f)}

OOS = {"CircadianRhythm","DynSysDelay","IkedaDelay",
       "PiecewiseCircuit","ScrollDelay","SprottDelay"}

in_scope = {k: v for k, v in rows.items() if k not in OOS}
hds      = {k: float(v["HD"]) for k, v in in_scope.items()}

# Sorted list of in-scope system names by HD (ascending)
systems_sorted = sorted(in_scope.keys(), key=lambda s: hds[s])

print(f"In-scope systems: {len(in_scope)}")
print(f"Median HD: {statistics.median(hds.values()):.3f}")


# ═══════════════════════════════════════════════════════════════════════════════
# Helper: load NPZ
# ═══════════════════════════════════════════════════════════════════════════════
def load_npz(system):

    path = Path(NPZ_DIR) / f"{system}.npz"

    if not path.exists():
        return None, None

    data = np.load(path, allow_pickle=True)

    gt = data["truth"]
    recon = data["pred"]

    return gt, recon


def best_2d_projection(pts, prefer="xy"):
    """Return indices of two axes with highest variance."""
    if pts is None or pts.ndim < 2 or pts.shape[1] < 2:
        return 0, 1
    variances = pts.var(axis=0)
    top2 = np.argsort(variances)[::-1][:2]
    return int(top2[0]), int(top2[1])


AXIS_LABELS = ["x", "y", "z", "w", "v", "u"]


def axis_label(i):
    return AXIS_LABELS[i] if i < len(AXIS_LABELS) else f"x_{i}"


# ═══════════════════════════════════════════════════════════════════════════════
# 1. HD distribution: histogram + CDF
# ═══════════════════════════════════════════════════════════════════════════════
def plot_hd_distribution(hds_dict, out_path):
    vals = sorted(hds_dict.values())
    n    = len(vals)

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

    # Histogram (log x-axis)
    ax = axes[0]
    log_vals = [math.log10(v) for v in vals]
    bins = np.linspace(min(log_vals) - 0.1, max(log_vals) + 0.1, 30)
    ax.hist(log_vals, bins=bins, color="#2E86AB", edgecolor="white", linewidth=0.4, alpha=0.85)
    for thresh, label in [(0.1, "HD=0.1"), (1.0, "HD=1.0")]:
        ax.axvline(math.log10(thresh), color="#C0392B", lw=1.2, ls="--")
        ax.text(math.log10(thresh) + 0.05, ax.get_ylim()[1] * 0.92, label,
                color="#C0392B", fontsize=7, va="top")
    ax.set_xlabel("Hausdorff Distance (log₁₀ scale)")
    ax.set_ylabel("Number of systems")
    ax.set_title("Histogram of HD (130 in-scope systems)")
    # Custom x tick labels
    ticks = [-2, -1, 0, 1, 2]
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"$10^{{{t}}}$" for t in ticks])

    # CDF
    ax = axes[1]
    xs = [0.0] + vals + [vals[-1] * 1.05]
    ys = [0.0] + [(i + 1) / n for i in range(n)] + [1.0]
    ax.step(xs, ys, color="#2E86AB", lw=1.5, where="post")
    for thresh, label, yoff in [(0.1, "HD<0.1", 0.05), (1.0, "HD<1.0", 0.05)]:
        frac = sum(1 for v in vals if v < thresh) / n
        ax.axvline(thresh, color="#C0392B", lw=1.2, ls="--")
        ax.axhline(frac,   color="#C0392B", lw=0.8, ls=":", alpha=0.6)
        n_sys = sum(1 for v in vals if v < thresh)
        ax.text(thresh * 1.08, frac + yoff,
                f"{label}\n{n_sys} systems ({frac*100:.0f}%)",
                color="#C0392B", fontsize=7)
    ax.set_xscale("log")
    ax.set_xlabel("Hausdorff Distance")
    ax.set_ylabel("Cumulative fraction")
    ax.set_title("Empirical CDF of HD")
    ax.set_xlim(min(vals) * 0.7, max(vals) * 2)
    ax.set_ylim(-0.02, 1.05)

    # Shared annotation
    med = statistics.median(vals)
    q1  = statistics.quantiles(vals, n=4)[0]
    q3  = statistics.quantiles(vals, n=4)[2]
    fig.text(0.5, -0.02,
             f"Median HD = {med:.3f}  (IQR: {q1:.3f}–{q3:.3f})   |   "
             f"HD < 0.1: {sum(1 for v in vals if v < 0.1)}   "
             f"HD < 1.0: {sum(1 for v in vals if v < 1.0)}",
             ha="center", fontsize=8, color="#444")

    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_path}")


plot_hd_distribution(hds, f"{OUT_DIR}/hd_distribution.pdf")


# ═══════════════════════════════════════════════════════════════════════════════
# Helper: single 2D panel (GT + recon)
# ═══════════════════════════════════════════════════════════════════════════════
def draw_panel(ax, gt, recon, title, hd_val, max_gt_pts=3000, max_rc_pts=2000):
    """Draw GT (grey) and recon (red) in best 2D projection on ax."""
    if gt is None:
        ax.text(0.5, 0.5, "NPZ not found", ha="center", va="center",
                transform=ax.transAxes, color="red", fontsize=8)
        ax.set_title(title, fontsize=8)
        return

    i, j = best_2d_projection(gt)

    # Subsample for speed
    idx_gt = np.random.choice(len(gt), min(max_gt_pts, len(gt)), replace=False)
    ax.scatter(gt[idx_gt, i], gt[idx_gt, j],
               s=0.4, c=GT_COLOR, alpha=0.35, linewidths=0, rasterized=True,
               label="Ground truth")

    if recon is not None and len(recon) > 0:
        idx_rc = np.random.choice(len(recon), min(max_rc_pts, len(recon)), replace=False)
        ax.scatter(recon[idx_rc, i], recon[idx_rc, j],
                   s=1.2, c=RECON_COLOR, alpha=0.7, linewidths=0, rasterized=True,
                   label="Reconstructed")

    ax.set_xlabel(axis_label(i), labelpad=1)
    ax.set_ylabel(axis_label(j), labelpad=1)
    ax.set_title(f"{title}\n(HD = {hd_val:.3f})", fontsize=8, pad=3)
    ax.tick_params(length=2)


# ═══════════════════════════════════════════════════════════════════════════════
# 2. GT vs recon: 3 successful systems
# ═══════════════════════════════════════════════════════════════════════════════
SUCCESS_EXAMPLES = ["Lorenz", "SprottR", "Halvorsen"]   # diverse, well-known

def plot_gt_recon_panel(system_list, out_path, title_prefix=""):
    fig, axes = plt.subplots(1, len(system_list), figsize=(4.5 * len(system_list), 3.8))
    if len(system_list) == 1:
        axes = [axes]
    for ax, sysname in zip(axes, system_list):
        gt, recon = load_npz(sysname)
        hd = hds.get(sysname, float("nan"))
        draw_panel(ax, gt, recon, sysname, hd)

    # Legend (from first valid panel)
    handles = [
        plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=GT_COLOR,
                   markersize=5, label="Ground truth"),
        plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=RECON_COLOR,
                   markersize=5, label="Reconstructed"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=2, fontsize=8,
               frameon=False, bbox_to_anchor=(0.5, -0.06))
    fig.tight_layout(rect=[0, 0.05, 1, 1])
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_path}")


plot_gt_recon_panel(SUCCESS_EXAMPLES,
                    f"{OUT_DIR}/gt_recon_success.pdf")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. GT vs recon: failure examples
# ═══════════════════════════════════════════════════════════════════════════════
# One from each main failure category (excluding delay/OOS which have no NPZ)
FAILURE_EXAMPLES = ["HyperRossler", "DoubleGyre", "DoublePendulum"]

plot_gt_recon_panel(FAILURE_EXAMPLES,
                    f"{OUT_DIR}/gt_recon_failure.pdf")


# ═══════════════════════════════════════════════════════════════════════════════
# 4. Per-system GT vs recon (all 130 in-scope)
# ═══════════════════════════════════════════════════════════════════════════════
def plot_per_system(system, out_dir):
    gt, recon = load_npz(system)
    if gt is None:
        print(f"  [SKIP] {system}: NPZ not found")
        return
    hd = hds.get(system, float("nan"))
    status = in_scope[system]["Status"]

    fig, ax = plt.subplots(figsize=(4, 3.5))
    draw_panel(ax, gt, recon, system, hd)

    color = "#27AE60" if status == "SUCCESS" else "#E67E22" if status == "POOR" else "#C0392B"
    ax.text(0.98, 0.02, status, transform=ax.transAxes, ha="right", va="bottom",
            fontsize=7, color=color, fontweight="bold")

    handles = [
        plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=GT_COLOR,
                   markersize=4, label="GT"),
        plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=RECON_COLOR,
                   markersize=4, label="Recon"),
    ]
    ax.legend(handles=handles, fontsize=7, frameon=False, loc="upper left")
    fig.tight_layout()
    fig.savefig(f"{out_dir}/{system}.pdf", bbox_inches="tight")
    plt.close(fig)


print("\nGenerating per-system figures...")
for i, sysname in enumerate(systems_sorted):
    print(f"  [{i+1:3d}/{len(systems_sorted)}] {sysname}")
    plot_per_system(sysname, f"{OUT_DIR}/per_system")

print(f"\nPer-system figures saved to {OUT_DIR}/per_system/")


# ═══════════════════════════════════════════════════════════════════════════════
# 5. Mosaic: all 130 in-scope systems, sorted by HD
# ═══════════════════════════════════════════════════════════════════════════════
def plot_mosaic(systems_sorted_list, hds_dict, in_scope_dict, npz_dir,
                out_path, cols=12, max_pts=800):
    """
    Compact mosaic of 2D projections for all in-scope systems.
    sorted by HD ascending (best → worst).
    """
    n = len(systems_sorted_list)
    rows_n = math.ceil(n / cols)
    cell_size = 1.5
    fig = plt.figure(figsize=(cols * cell_size, rows_n * cell_size * 1.15))

    loaded = {}
    for sysname in systems_sorted_list:
        gt, recon = load_npz(sysname)
        loaded[sysname] = (gt, recon)

    for idx, sysname in enumerate(systems_sorted_list):
        ax = fig.add_subplot(rows_n, cols, idx + 1)
        gt, recon = loaded[sysname]
        hd = hds_dict[sysname]
        status = in_scope_dict[sysname]["Status"]

        if gt is None:
            ax.text(0.5, 0.5, "?", ha="center", va="center",
                    transform=ax.transAxes, fontsize=8)
        else:
            i, j = best_2d_projection(gt)
            idx_gt = (np.random.choice(len(gt), min(max_pts, len(gt)), replace=False)
                      if len(gt) > max_pts else np.arange(len(gt)))
            ax.scatter(gt[idx_gt, i], gt[idx_gt, j],
                       s=0.2, c=GT_COLOR, alpha=0.25, linewidths=0, rasterized=True)
            if recon is not None and len(recon) > 0:
                idx_rc = (np.random.choice(len(recon), min(max_pts, len(recon)), replace=False)
                          if len(recon) > max_pts else np.arange(len(recon)))
                ax.scatter(recon[idx_rc, i], recon[idx_rc, j],
                           s=0.5, c=RECON_COLOR, alpha=0.6, linewidths=0, rasterized=True)

        # Title: system name abbreviated + HD
        short = sysname[:10] + "…" if len(sysname) > 10 else sysname
        title_color = ("#27AE60" if status == "SUCCESS"
                       else "#E67E22" if status == "POOR" else "#C0392B")
        ax.set_title(f"{short}\n{hd:.2f}", fontsize=5, color=title_color, pad=1)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_linewidth(0.3)
            spine.set_color("#aaa")

    # Legend
    handles = [
        plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=GT_COLOR,
                   markersize=5, label="Ground truth"),
        plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=RECON_COLOR,
                   markersize=5, label="Reconstructed"),
        plt.Line2D([0],[0], marker='o', color='w', markerfacecolor="#27AE60",
                   markersize=5, label="SUCCESS"),
        plt.Line2D([0],[0], marker='o', color='w', markerfacecolor="#E67E22",
                   markersize=5, label="POOR"),
        plt.Line2D([0],[0], marker='o', color='w', markerfacecolor="#C0392B",
                   markersize=5, label="EXPLODED/FAILED"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=5, fontsize=7,
               frameon=False, bbox_to_anchor=(0.5, -0.01))

    fig.suptitle(
        f"HMKF Benchmark: All {n} In-Scope Systems  (sorted by Hausdorff Distance, ascending)",
        fontsize=9, y=1.005
    )
    fig.tight_layout(pad=0.3)
    fig.savefig(out_path, bbox_inches="tight", dpi=200)
    plt.close(fig)
    print(f"Saved mosaic: {out_path}")


print("\nGenerating full mosaic...")
plot_mosaic(
    systems_sorted_list=systems_sorted,
    hds_dict=hds,
    in_scope_dict=in_scope,
    npz_dir=NPZ_DIR,
    out_path=f"{OUT_DIR}/full_benchmark_mosaic.pdf",
    cols=13,
)

print(f"\n✓ All figures written to {OUT_DIR}/")
print("  hd_distribution.pdf")
print("  gt_recon_success.pdf")
print("  gt_recon_failure.pdf")
print("  full_benchmark_mosaic.pdf")
print("  per_system/{system}.pdf  (130 files)")

In-scope systems: 130
Median HD: 0.323
Saved: /content/drive/MyDrive/HMKF_Clean/figures_revised/hd_distribution.pdf
Saved: /content/drive/MyDrive/HMKF_Clean/figures_revised/gt_recon_success.pdf
Saved: /content/drive/MyDrive/HMKF_Clean/figures_revised/gt_recon_failure.pdf

Generating per-system figures...
  [  1/130] PanXuZhou
  [  2/130] CaTwoPlus
  [  3/130] CellularNeuralNetwork
  [  4/130] Blasius
  [  5/130] LorenzBounded
  [  6/130] RayleighBenard
  [  7/130] SprottJ
  [  8/130] BeerRNN
  [  9/130] SprottS
  [ 10/130] Rucklidge
  [ 11/130] ShimizuMorioka
  [ 12/130] Bouali2
  [ 13/130] QiChen
  [ 14/130] BurkeShaw
  [ 15/130] Lorenz84
  [ 16/130] Lorenz
  [ 17/130] AnishchenkoAstakhov
  [ 18/130] SprottQ
  [ 19/130] Tsucs2
  [ 20/130] Sakarya
  [ 21/130] Halvorsen
  [ 22/130] VallisElNino
  [ 23/130] HindmarshRose
  [ 24/130] Rossler
  [ 25/130] IsothermalChemical
  [ 26/130] Dadras
  [ 27/130] LuChen
  [ 28/130] SprottG
  [ 29/130] SprottR
  [ 30/130] LorenzStenflo
  [ 31/130] Ha

In [ ]:
import statistics

all_hds = [float(r["HD"]) for r in rows.values()]
print("Median (all):", statistics.median(all_hds))

in_scope_hds = [float(r["HD"]) for r in in_scope.values()]
print("Median (in-scope):", statistics.median(in_scope_hds))

Median (all): 0.41699646251138983
Median (in-scope): 0.3225986543109669


In [ ]:
# Save active kernels figure
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv(RESULTS_CSV)
OOS = {"CircadianRhythm","DynSysDelay","IkedaDelay",
       "PiecewiseCircuit","ScrollDelay","SprottDelay"}
df = df[~df['System'].isin(OOS)].copy()

fig, ax = plt.subplots(figsize=(5, 3.8))
for status, color, marker in [("SUCCESS", "#27AE60", "o"),
                                ("POOR",    "#E67E22", "s"),
                                ("EXPLODED","#C0392B", "^"),
                                ("FAILED",  "#C0392B", "^")]:
    sub = df[df['Status'] == status]
    ax.scatter(sub['Active'], sub['HD'],
               c=color, marker=marker, s=18, alpha=0.7,
               linewidths=0, label=status, rasterized=True)

ax.set_yscale("log")
ax.set_xlabel("Number of active kernels")
ax.set_ylabel("Hausdorff Distance (log scale)")
ax.set_title("Active kernels vs reconstruction error (130 systems)")
ax.legend(fontsize=8, frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()
fig.savefig(f"{OUT_DIR}/active_kernels.pdf", bbox_inches="tight")
plt.close(fig)
print("Saved: active_kernels.pdf")

Saved: active_kernels.pdf
